# Inheritance Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Your first subclass.** The child wrote no `__init__` and no `introduce` — it simply inherited them.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def introduce(self):
        return f"I'm {self.name}, {self.age} years old."


class Student(Person):              # parent goes in parentheses
    def enroll(self, course):       # NEW ability the parent never had
        return f"{self.name} enrolled in {course}"


s = Student("Sarah", 22)
print(s.introduce())                # inherited
print(s.enroll("Linear Algebra"))   # brand new
print(isinstance(s, Person))        # a Student IS-A Person

**2. Free machinery.** A child with an empty body still carries every parent attribute and method.

In [ ]:
class Vehicle:
    def __init__(self, brand):
        self.brand = brand

    def describe(self):
        return f"{self.brand} rolls on wheels"


class Motorcycle(Vehicle):
    """Nothing added yet - pure inheritance."""
    pass


m = Motorcycle("Honda")
print(m.describe())                 # came from Vehicle, untouched
print(isinstance(m, Vehicle))

**3. Extend the constructor with `super()`.** A child `__init__` replaces the parent's entirely — `super().__init__()` hands the shared setup back up.

In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary


class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)   # parent does ITS part first
        self.team_size = team_size       # then the child adds its own


m = Manager("Amina", 90_000, 6)
print(m.name, m.salary, m.team_size)

## Part 2 — Practice

**4. Override, then extend.** Redefining the name replaces the parent version; `super().speak()` lets the child build on it instead of repeating it.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."


class Dog(Animal):
    def speak(self):                    # OVERRIDE
        return f"{self.name}: Woof!"


class RobotDog(Dog):
    def speak(self):                    # EXTEND: reuse Dog's answer
        return f"[beep] {super().speak()}"


for speaker in (Animal("Mystery"), Dog("Rex"), RobotDog("Unit-Rex")):
    print(speaker.speak())

**5. The forgotten super.** Defining `__init__` in the child REPLACES the parent's — without `super().__init__()` the parent attributes simply never exist.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age


class BadStudent(Person):
    def __init__(self, student_id):
        self.student_id = student_id     # Person.__init__ never ran!


bad = BadStudent("X-001")
try:
    print(bad.name)
except AttributeError as err:
    print("AttributeError:", err)


class GoodStudent(Person):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)      # FIXED: parent sets up first
        self.student_id = student_id


good = GoodStudent("Rafi", 19, "CSE-2026-041")
print(good.name, good.age, good.student_id)

**6. Who answers? Climb the MRO.** Lookup walks Puppy, then Dog, then Animal — and stops at the first hit.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def info(self):
        return f"{self.name}: {self.speak()}"


class Dog(Animal):
    def speak(self):
        return "Woof!"


class Puppy(Dog):
    def fetch(self):                    # exists ONLY at this level
        return f"{self.name} brings the ball!"


pip = Puppy("Pip")

print(pip.fetch())    # found ON Puppy
print(pip.speak())    # Puppy lacks it -> found on Dog
print(pip.info())     # Puppy and Dog lack it -> found on Animal
print(pip.name)       # attributes climb the same ladder
print([c.__name__ for c in Puppy.__mro__])

**7. A smarter list.** Built-ins are ordinary classes — legitimate parents whose entire toolbox you keep.

In [ ]:
class ScoreList(list):
    """Exam scores that know their own statistics."""

    def average(self):
        return sum(self) / len(self)

    def top(self):
        return max(self) if self else None


scores = ScoreList([88, 92, 79])

scores.append(95)               # native methods still work
scores.sort()
print(scores)
print(scores.average(), scores.top())
print(ScoreList([]).top())      # edge case handled
print(isinstance(scores, list)) # a ScoreList IS-A list

## Part 3 — Challenge

**8. Two parents, one winner.** The MRO is a single left-to-right linearization — ties go to the earlier parent, and each ancestor is visited exactly once.

In [ ]:
class Swimmer:
    def move(self):
        return "swimming"


class Flyer:
    def move(self):
        return "flying"


class Duck(Swimmer, Flyer):        # left parent wins ties
    pass


d = Duck()
print(d.move())                               # predicted: swimming
print([c.__name__ for c in Duck.__mro__])


class A:
    def hello(self):
        return "A"


class B(A):
    def hello(self):
        return "B"


class C(A):
    def hello(self):
        return "C"


class D(B, C):                     # D sees A twice through two paths
    pass


print(D().hello())                            # predicted: B
print([c.__name__ for c in D.__mro__])        # each ancestor exactly once

**9. Has-a beats is-a.** Composition stores the part as an attribute and delegates — parts become swappable without touching the whole.

In [ ]:
class PetrolEngine:
    def start(self):
        return "Vroom! Engine running."


class ElectricEngine:
    def start(self):
        return "...silent hum..."


class Car:
    def __init__(self, model, engine):
        self.model = model
        self.engine = engine            # Car HAS-A engine

    def start(self):
        return f"{self.model}: {self.engine.start()}"


print(Car("BMW M3", PetrolEngine()).start())
print(Car("Tesla Model 3", ElectricEngine()).start())
# With inheritance the engine choice would be frozen into the class tree;
# composition lets ANY engine plug in at construction time.